In [1]:
import torch 
from sklearn.datasets import load_iris
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
iris = load_iris()

In [3]:
X = iris.data
y = iris.target.reshape(-1, 1)

encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

X_train_np, X_test_np, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [4]:
torch.manual_seed(42)

intializing weights and bias

In [5]:
#hidden layer 1
W1 = torch.randn(4,3) *0.01 #as the weights initialized are quite large 
B1 = torch.zeros(1,3) #dim is (1,3) not (4,3) as bias is per neuron and weight is per pair of neuron and input feature
#we set weights as random values so as to break the symmetry , whereas similar bias does not effect the learning of neurons

#hidden layer 2
W2 = torch.randn(3,3) *0.01 
B2 = torch.zeros(1,3) 

#outpur layer 
W3 = torch.randn(3,3) *0.01
B3 = torch.zeros(1,3)

**Activation Function**

In [6]:
def relu(x):
    return torch.maximum(x, torch.zeros_like(x))

In [7]:
def relu_grad(x):
    return (x>0).float()

**softmax function**

In [8]:
def softmax(x):
    expX = torch.exp(x - torch.max(x, dim=1, keepdim=True).values)
    return expX/(torch.sum(expX,dim=1, keepdim=True))

**crossEntropy**

In [9]:
def crossEntropy(y_pred, y_true):
    lambda_=1e-10

    y_pred = torch.clamp(y_pred, lambda_, 1-lambda_)

    loss = -(torch.mean(torch.sum(y_true*torch.log(y_pred), dim=1)))
    return loss

**forwardLayer**

In [10]:
def forward(X):
    Z1 = X @ W1 + B1
    A1 = relu(Z1)

    Z2 = A1 @ W2 + B2
    A2 = relu(Z2)

    Z3 = A2 @ W3 + B3
    A3 = softmax(Z3)

    return A1, A2, A3, Z1, Z2, Z3

**training**

In [11]:
epochs = 2000
lr = 0.1

for i in range(epochs):

    #forward pass
    A1,A2,A3,Z1,Z2,Z3 = forward(X_train)

    #crossEntropy 
    loss = crossEntropy(A3, y_train)
    
    #output layer
    dZ3 = A3 - y_train 

    dW3 = (A2.T @ dZ3)/(X_train.shape[0])
    dB3 = torch.sum(dZ3, dim=0, keepdim=True)/X_train.shape[0]

    #hidden layer 2
    dA2 = dZ3 @ W3.T
    dZ2 = dA2 * relu_grad(Z2)

    dW2 = (A1.T @ dZ2)/(X_train.shape[0])
    dB2 = torch.sum(dZ2, dim=0, keepdim=True)/X_train.shape[0]

    #hidden layer 1
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_grad(Z1)

    dW1 = (X_train.T @ dZ1)/(X_train.shape[0])
    dB1 = torch.sum(dZ1, dim=0, keepdim=True)/X_train.shape[0]

    #updation
    W1 -= lr*dW1
    W2 -= lr*dW2
    W3 -= lr*dW3

    B1 -= lr*dB1
    B2 -= lr*dB2
    B3 -= lr*dB3

    #track loss 
    if i%100==0:
        print(f"loss(iter:{i}) : {loss}")

loss(iter:0) : 1.0986121892929077
loss(iter:100) : 1.098404049873352
loss(iter:200) : 1.0984036922454834
loss(iter:300) : 1.0984036922454834
loss(iter:400) : 1.0984035730361938
loss(iter:500) : 1.0984036922454834
loss(iter:600) : 1.0984035730361938
loss(iter:700) : 1.0984034538269043
loss(iter:800) : 1.0984034538269043
loss(iter:900) : 1.0984032154083252
loss(iter:1000) : 1.0984028577804565
loss(iter:1100) : 1.0984026193618774
loss(iter:1200) : 1.0984023809432983
loss(iter:1300) : 1.0984019041061401
loss(iter:1400) : 1.0984013080596924
loss(iter:1500) : 1.098400354385376
loss(iter:1600) : 1.098399043083191
loss(iter:1700) : 1.098396897315979
loss(iter:1800) : 1.0983937978744507
loss(iter:1900) : 1.098388433456421


In [12]:
_,_,prediction,_,_,_ = forward(X_test)

class_pred = torch.argmax(prediction, dim=1)
class_true = torch.argmax(y_test, dim=1)

#accuracy calculation:
accuracy = (class_pred==class_true).float().mean()

print("Accuracy:", accuracy.item())

Accuracy: 0.30000001192092896
